El script agrupa todas las fuentes de SIUC en un .parquet y genera una versión estructurada de la base de conectados en formato .parquet.

In [2]:
import os
import pandas as pd
import pyarrow

# SIUC

In [23]:
# Get all sheet names
excel_file = 'data/sinformato/siuc_asistencias.xlsx'
xls = pd.ExcelFile(excel_file)
sheet_names = xls.sheet_names

# Create empty list to store DataFrames
dfs = []

# Read each sheet and append to list
for sheet in sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet)
    dfs.append(df)

# Concatenate all DataFrames into one
df_final = pd.concat(dfs, ignore_index=True)
df_final

,Id,Identificacion (Asistente),Nombre del alumno,Solicitante,Genero,Region,Ciudad,Empresa del estudiante,Area Gerencia,Cons agenda,...,Cons solicitud,Nombre de la solicitud,Ejecutor,Indicador a impactar,Cons sesion,Temas dictados,Agendador,Tiempo dedicado al entrenamiento,Estado o proceso del entrenamiento,Empresa del ejecutivo
0,127207,72264404,Jorge Eliecer Cantillo Ibañez,Día 1 R2,Masculino,R2,Medellin-R2,OVERLAP,AGENTES,5815,...,1714,Día 1 R2,Universidad Claro,"Adiciones Netas, Altas, Calidad de la venta, G...",7469,CONECTADOS,LUNA CAMILA HERNANDEZ ORTIZ,02:00:00,Finalizado,CLARO
1,127207,72264404,Jorge Eliecer Cantillo Ibañez,Día 1 R2,Masculino,R2,Medellin-R2,OVERLAP,AGENTES,5815,...,1714,Día 1 R2,Universidad Claro,"Adiciones Netas, Altas, Calidad de la venta, G...",7469,OFERTA FÉNIX,LUNA CAMILA HERNANDEZ ORTIZ,02:00:00,Finalizado,CLARO
2,127207,72264404,Jorge Eliecer Cantillo Ibañez,Día 1 R2,Masculino,R2,Medellin-R2,OVERLAP,AGENTES,5815,...,1714,Día 1 R2,Universidad Claro,"Adiciones Netas, Altas, Calidad de la venta, G...",7469,OFERTA INFALTABLE,LUNA CAMILA HERNANDEZ ORTIZ,02:00:00,Finalizado,CLARO
3,127207,72264404,Jorge Eliecer Cantillo Ibañez,Día 1 R2,Masculino,R2,Medellin-R2,OVERLAP,AGENTES,5815,...,1714,Día 1 R2,Universidad Claro,"Adiciones Netas, Altas, Calidad de la venta, G...",7469,PLANES POSPAGOS,LUNA CAMILA HERNANDEZ ORTIZ,02:00:00,Finalizado,CLARO
4,127207,72264404,Jorge Eliecer Cantillo Ibañez,Día 1 R2,Masculino,R2,Medellin-R2,OVERLAP,AGENTES,5815,...,1714,Día 1 R2,Universidad Claro,"Adiciones Netas, Altas, Calidad de la venta, G...",7469,TOP OF MIND,LUNA CAMILA HERNANDEZ ORTIZ,02:00:00,Finalizado,CLARO
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1713150,770118,36497104,NORLEIDA MANDON SUAREZ,Ponte al día con la U semana del 27 al 31 de O...,Femenino,R4,Bogota-R4,OVERLAP,CAV,22763,...,4905,Ponte al día con la U semana del 27 al 31 de O...,Universidad Claro,"Altas, Churn, GANA, NPS",29560,CAMPAÑA TODO CLARO - POLIEDRO,María de los Angeles Cárdenas Prieto,00:30:00,Finalizado,CLARO
1713151,770118,36497104,NORLEIDA MANDON SUAREZ,Ponte al día con la U semana del 27 al 31 de O...,Femenino,R4,Bogota-R4,OVERLAP,CAV,22763,...,4905,Ponte al día con la U semana del 27 al 31 de O...,Universidad Claro,"Altas, Churn, GANA, NPS",29560,CRÉDITO TU DESVARE,María de los Angeles Cárdenas Prieto,00:30:00,Finalizado,CLARO
1713152,770139,1121902507,Angie paola Forero Perez,Cuartil EST-MAS-PRE-CCO-1020,Femenino,R5,Villavicencio-R5,OVERLAP,AGENTES,20958,...,4697,Cuartil EST-MAS-PRE-CCO-1020,Universidad Claro,Altas,29184,PRESENTACIÓN / ARGUMENTACIÓN,LUNA CAMILA HERNANDEZ ORTIZ,06:00:00,Finalizado,MOVILCO SAS + AGENTES COMERCIALES
1713153,770140,40332596,Claudia Maritza Sierra Castillo,Cuartil EST-MAS-PRE-CCO-1020,Femenino,R5,Villavicencio-R5,OVERLAP,AGENTES,20958,...,4697,Cuartil EST-MAS-PRE-CCO-1020,Universidad Claro,Altas,29184,PRESENTACIÓN / ARGUMENTACIÓN,LUNA CAMILA HERNANDEZ ORTIZ,06:00:00,Finalizado,MOVILCO SAS + AGENTES COMERCIALES


In [49]:
# Esta linea de codigo es clave para evitar problemas al importar a Databricks
df_final = df_final.astype({col: "string" for col in df_final.select_dtypes(include=["object"]).columns})

df_final = df_final.astype({
    "Id": "string",
    "Identificacion (Asistente)": "string"
})



In [51]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1713155 entries, 0 to 1713154
Data columns (total 26 columns):
 #   Column                              Dtype 
---  ------                              ----- 
 0   Id                                  string
 1   Identificacion (Asistente)          string
 2   Nombre del alumno                   string
 3   Solicitante                         string
 4   Genero                              string
 5   Region                              string
 6   Ciudad                              string
 7   Empresa del estudiante              string
 8   Area Gerencia                       string
 9   Cons agenda                         int64 
 10  Coordinacion                        string
 11  Ubicacion                           string
 12  Ejecutivo                           string
 13  Metodologia                         string
 14  Fecha real de creacion asistencia   string
 15  Operacion                           string
 16  Cons solicitud    

In [52]:
# Exportar con pyarrow forzando timestamps a ms
df_final.to_parquet(
    'data/listos_para_cargar_databricks/siuc_asistencias.parquet',
    engine='pyarrow',
    index=False,
    coerce_timestamps='ms',
    allow_truncated_timestamps=True
)
print("DataFrame exported successfully to Parquet format")

DataFrame exported successfully to Parquet format


In [44]:
# Leer y verificar
df_test = pd.read_parquet('data/listos_para_cargar_databricks/siuc_asistencias.parquet', engine='pyarrow')
print("Shape:", df_test.shape)
print(df_test.dtypes)

Shape: (1713155, 26)
Id                                             int64
Identificacion (Asistente)                     int64
Nombre del alumno                     string[python]
Solicitante                           string[python]
Genero                                string[python]
Region                                string[python]
Ciudad                                string[python]
Empresa del estudiante                string[python]
Area Gerencia                         string[python]
Cons agenda                                    int64
Coordinacion                          string[python]
Ubicacion                             string[python]
Ejecutivo                             string[python]
Metodologia                           string[python]
Fecha real de creacion asistencia     string[python]
Operacion                             string[python]
Cons solicitud                                 int64
Nombre de la solicitud                string[python]
Ejecutor                 

In [4]:
conectados = pd.read_csv('data/listos_para_cargar_databricks/conectados_ingresos.csv')

In [5]:
# Exportar con pyarrow forzando timestamps a ms
conectados.to_parquet(
    'data/listos_para_cargar_databricks/consumo_conectados.parquet',
    engine='pyarrow',
    index=False,
    coerce_timestamps='ms',
    allow_truncated_timestamps=True
)
print("DataFrame exported successfully to Parquet format")

DataFrame exported successfully to Parquet format


# Conectados

In [ ]:
# ...existing code...
import os
import pandas as pd
import math
import time
from typing import List, Tuple

def _progress_bar(pct: float, width: int = 30) -> str:
    filled = int(math.floor(max(0, min(1, pct)) * width))
    bar = '█' * filled + '-' * (width - filled)
    return f'|{bar}| {pct*100:5.1f}%'

def scan_conectados_files(base_dir: str,
                          subfolders: List[str] = None) -> Tuple[List[Tuple[str,str,str,int]], int]:
    """
    Escanea las subcarpetas y devuelve un listado (folder, fname, ext, sheet_count)
    y el total de unidades a procesar (cada hoja de excel cuenta como unidad).
    """
    if subfolders is None:
        subfolders = ['Articulos', 'Base Oferta', 'Comunicados']
    accepted_exts = {'.csv', '.xlsx', '.xls', '.parquet', '.json'}

    file_map = []
    total_units = 0

    for folder in subfolders:
        folder_path = os.path.join(base_dir, folder)
        if not os.path.isdir(folder_path):
            print(f"[SKIP] No existe: {folder_path}", flush=True)
            continue
        for fname in sorted(os.listdir(folder_path)):
            fpath = os.path.join(folder_path, fname)
            if os.path.isdir(fpath):
                continue
            _, ext = os.path.splitext(fname.lower())
            if ext not in accepted_exts:
                print(f"[IGNORADO] Tipo no soportado: {fpath}", flush=True)
                continue
            sheet_count = 1
            if ext in ('.xlsx', '.xls'):
                try:
                    xls = pd.ExcelFile(fpath)
                    sheet_count = len(xls.sheet_names)
                except Exception as e:
                    print(f"[AVISO] No se pudo inspeccionar hojas de {fname}: {e}", flush=True)
                    sheet_count = 1
            file_map.append((folder, fname, ext, sheet_count))
            total_units += sheet_count

    print(f"\nResumen escaneo: {len(file_map)} archivos detectados, {total_units} unidades (hojas/archivos).", flush=True)
    return file_map, total_units

def read_and_collect(base_dir: str,
                     file_map: List[Tuple[str,str,str,int]],
                     total_units: int,
                     verbose: bool = True) -> List[pd.DataFrame]:
    """
    Lee cada archivo/hoja, añade columnas 'Tipo de Contenido' y 'Archivo',
    devuelve la lista de dataframes y muestra progreso por unidad.
    """
    dfs = []
    processed_units = 0
    start_time = time.time()

    for folder, fname, ext, sheet_count in file_map:
        fpath = os.path.join(base_dir, folder, fname)
        if verbose:
            print(f"\n[LEER] {fpath}  (ext: {ext})", flush=True)
        try:
            if ext == '.csv':
                df_list = [pd.read_csv(fpath, dtype=str, keep_default_na=False)]
            elif ext in ('.xlsx', '.xls'):
                sheets = pd.read_excel(fpath, sheet_name=None)  # dict: hoja -> df
                df_list = list(sheets.values())
            elif ext == '.parquet':
                df_list = [pd.read_parquet(fpath)]
            elif ext == '.json':
                df_list = [pd.read_json(fpath)]
            else:
                df_list = []
        except Exception as e:
            print(f"[ERROR] Leyendo {fpath}: {e}", flush=True)
            # contabilizar hojas aunque fallen para mantener progreso aproximado
            processed_units += sheet_count
            continue

        for idx, df in enumerate(df_list, start=1):
            # Asegurar copia y convertir todas las columnas a string (pandas StringDtype)
            try:
                df = df.copy()
                # convertir todo a string para evitar conversiones problemáticas al escribir parquet
                df = df.astype("string")
            except Exception:
                # fallback: forzar str genérico
                df = df.applymap(lambda x: "" if pd.isna(x) else str(x))

            # insertar columnas al inicio
            df.insert(0, 'Tipo de Contenido', folder)
            df.insert(1, 'Archivo', fname)

            dfs.append(df)
            processed_units += 1

            if verbose:
                pct = processed_units / total_units if total_units else 1.0
                bar = _progress_bar(pct)
                elapsed = time.time() - start_time
                print(f"  Unidad {processed_units}/{total_units} ({idx}/{sheet_count}) {bar} — {elapsed:.1f}s", flush=True)

    return dfs

def finalize_and_write(dfs: List[pd.DataFrame],
                       output_path: str = 'data/listos_para_cargar_databricks/conectados_combinado.parquet'):
    """
    Concatena, normaliza tipos a pandas StringDtype y escribe .parquet.
    Si falla pyarrow por tipos conflictivos, reintenta convirtiendo todo a string puro.
    """
    if not dfs:
        print("No hay dataframes para concatenar.", flush=True)
        return None

    df_final = pd.concat(dfs, ignore_index=True)
    print(f"\nConcatenado final — filas: {len(df_final)}, columnas: {len(df_final.columns)}", flush=True)

    # forzar columnas objeto -> pandas StringDtype
    try:
        obj_cols = df_final.select_dtypes(include=['object']).columns.tolist()
        if obj_cols:
            df_final = df_final.astype({col: "string" for col in obj_cols})
    except Exception as e:
        print(f"[AVISO] No se pudieron convertir columnas object a StringDtype: {e}", flush=True)

    # intentar escribir
    try:
        df_final.to_parquet(output_path,
                            engine='pyarrow',
                            index=False,
                            coerce_timestamps='ms',
                            allow_truncated_timestamps=True)
        print(f"[OK] Archivo escrito: {output_path}", flush=True)
    except Exception as e:
        print(f"[ERROR] Escritura parquet falló: {e}", flush=True)
        print("[REINTENTO] Convirtiendo todas las columnas a string y reescribiendo...", flush=True)
        try:
            df_final = df_final.astype("string")
            df_final.to_parquet(output_path,
                                engine='pyarrow',
                                index=False,
                                coerce_timestamps='ms',
                                allow_truncated_timestamps=True)
            print(f"[OK] Archivo escrito: {output_path} (todos como string)", flush=True)
        except Exception as e2:
            print(f"[FATAL] Reintento fallido: {e2}", flush=True)
            raise

    return df_final


In [6]:
base_dir = 'data/sinformato/Conectados'   

# 1) listar y contar unidades
file_map, total_units = scan_conectados_files(base_dir)

# 2) leer y recolectar (muestra progreso por unidad)
dfs = read_and_collect(base_dir, file_map, total_units, verbose=True)

# 3) concatenar y escribir parquet (manejo de errores de tipo)
df_final = finalize_and_write(dfs,
                              output_path='data/listos_para_cargar_databricks/conectados_combinado.parquet')

# 4) comprobación final
if df_final is not None:
    print("\nResumen final:", flush=True)
    print("Shape:", df_final.shape, flush=True)
    display(df_final.head(3))
# ...existing code...


Resumen escaneo: 69 archivos detectados, 70 unidades (hojas/archivos).

[LEER] data/sinformato/Conectados\Articulos\01.04.2025 articulos.xlsx  (ext: .xlsx)
  Unidad 1/70 (1/1) |------------------------------|   1.4% — 1.6s

[LEER] data/sinformato/Conectados\Articulos\01.07.2025 articulos.xlsx  (ext: .xlsx)
  Unidad 2/70 (1/1) |------------------------------|   2.9% — 6.0s

[LEER] data/sinformato/Conectados\Articulos\03.03.2025 articulos.xlsx  (ext: .xlsx)
  Unidad 3/70 (1/1) |█-----------------------------|   4.3% — 6.1s

[LEER] data/sinformato/Conectados\Articulos\07.04.2025 articulos.xlsx  (ext: .xlsx)
  Unidad 4/70 (1/1) |█-----------------------------|   5.7% — 8.4s

[LEER] data/sinformato/Conectados\Articulos\07.07.2025 articulos.xlsx  (ext: .xlsx)
  Unidad 5/70 (1/1) |██----------------------------|   7.1% — 13.5s

[LEER] data/sinformato/Conectados\Articulos\08.09.2025 articulos.xlsx  (ext: .xlsx)
  Unidad 6/70 (1/1) |██----------------------------|   8.6% — 25.5s

[LEER] data/s

,Tipo de Contenido,Archivo,id,cedula,web,url,fecha_hora
0,Articulos,01.04.2025 articulos.xlsx,1,1117816914,Claro_TV,https://conectados.com.co/inicio/-/knowledge_b...,2025-02-06 11:13:49
1,Articulos,01.04.2025 articulos.xlsx,2,1033728706,Claro_TV,https://conectados.com.co/inicio/-/knowledge_b...,2025-02-06 11:14:04
2,Articulos,01.04.2025 articulos.xlsx,3,1117816914,Claro_TV,https://conectados.com.co/inicio/-/knowledge_b...,2025-02-06 11:17:33


In [8]:
df_final["Tipo de Contenido"].value_counts()

Tipo de Contenido
Base Oferta    6883091
Articulos      4219144
Comunicados    1370511
Name: count, dtype: Int64

# Categorización Certificación UMC

In [1]:
import pandas as pd
excel_file = 'data/sinformato/manual_categorizacion_certificacion_umc.xlsx'
categorizacion_cert = pd.read_excel(excel_file)

In [ ]:
categorizacion_cert.to_parquet(
'data/listos_para_cargar_databricks/manual_categorizacion_certificacion_umc.parquet',
engine='pyarrow',
index=False,
coerce_timestamps='ms',
allow_truncated_timestamps=True
)
print("DataFrame exported successfully to Parquet format")
categorizacion_cert.head(2)

# Demanda

In [7]:
import pandas as pd
excel_file = 'data/listos_para_cargar_databricks/human_demanda_umc.xlsx'
demanda = pd.read_excel(excel_file)
# Convertir todas las columnas object a pandas StringDtype
demanda = demanda.astype({col: "string" for col in demanda.select_dtypes(include=["object"]).columns})
demanda.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 265 entries, 0 to 264
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Año                            265 non-null    int64         
 1   Mes                            265 non-null    string        
 2   Estado                         265 non-null    string        
 3   Motivo                         265 non-null    string        
 4   Fecha de Ingreso de Solicitud  265 non-null    string        
 5   Planificado / Demanda          265 non-null    string        
 6   Fecha                          265 non-null    datetime64[ns]
 7   Diferencia Fechas              252 non-null    float64       
 8   Siuc                           263 non-null    string        
 9   Categoria                      265 non-null    string        
 10  Segmento                       265 non-null    string        
 11  Canal              

In [8]:
demanda.Mes.value_counts()

Mes
Octubre       95
Noviembre     79
Septiembre    55
Diciembre     36
Name: count, dtype: Int64

In [9]:
demanda.to_parquet(
'data/listos_para_cargar_databricks/human_demanda_umc.parquet',
engine='pyarrow',
index=False,
coerce_timestamps='ms',
allow_truncated_timestamps=True
)
print("DataFrame exported successfully to Parquet format")
demanda.head(2)

DataFrame exported successfully to Parquet format


,Año,Mes,Estado,Motivo,Fecha de Ingreso de Solicitud,Planificado / Demanda,Fecha,Diferencia Fechas,Siuc,Categoria,...,Estado KPI,Meta KPI,Hora,Público Objetivo,Modalidad,Lugar,Regional,Objetivo de la Sesión,Entrenador Asignado,Motivo Cancelación
0,2025,Septiembre,Ejecutado,Acompañamiento,2025-09-01 00:00:00,Planificado,2025-09-01,0.0,<NA>,Habilidades,...,<NA>,<NA>,8:00am - 6:00pm,<NA>,Presencial,306,R4,<NA>,Camilo Jaramillo,<NA>
1,2025,Septiembre,Ejecutado,Acompañamiento,2025-09-02 00:00:00,Planificado,2025-09-02,0.0,4360,Habilidades,...,<NA>,<NA>,8:00am - 6:00pm,<NA>,Presencial,307,R4,<NA>,Camilo Jaramillo,<NA>


# Maestro Personas

In [7]:
import pandas as pd
excel_file = 'data/sinformato/ssff_maestropersonas.xlsx'
personas = pd.read_excel(excel_file)

In [8]:
personas.to_parquet(
'data/listos_para_cargar_databricks/ssff_maestropersonas.parquet',
engine='pyarrow',
index=False,
coerce_timestamps='ms',
allow_truncated_timestamps=True
)
print("DataFrame exported successfully to Parquet format")
personas.head(2)

DataFrame exported successfully to Parquet format


,ID de sistema de usuario,Nombre de Usuario,Nombres,Apellidos,Nombre Cargo,Personal Información sobre el teléfono Número de teléfono,GV-Zona,Ciudad,Departamento,Género,...,Gerencia / Dirección Nombre,Dirección Comité Nombre,Estado de colaborador,Cédula,Corporativo Información sobre el correo electrónico Correo,Ubicación,Empresa Código,Corporativo Información sobre el teléfono Número de teléfono,Personal Información sobre el correo electrónico Correo,Colombia Cédula de ciudadanía DOCUMENTO DE IDENTIFICACION
0,45051291,1040750865,JHONATAN STEVEN,GUTIERREZ GALLO,Tecnico Instalaciones Bidireccional,NaN,NaN,73818,Antioquia,NaN,...,Aliados,(ING) Direccion Corporativa Tecnologia,Inactivo,1040750865,NaN,6100210-SEDE ALIADO,CA13,NaN,NaN,NaN
1,45059792,77838807,LUIS ERNESTO,SALANDY SOLANO,ASESOR TMK INBOUND,NaN,NaN,73260,"Bogotá, D.C.",NaN,...,Direccion Ventas Hogares,(HOG) Unidad Negocio Hogares,Inactivo,77838807,NaN,6100210-SEDE ALIADO,CA314,NaN,NaN,NaN


# Categorias auxiliares SSFF Certificaciones

In [3]:
import pandas as pd
excel_file = 'data/sinformato/aux_categ_cert_umc.xlsx'
aux_categ_cert_umc = pd.read_excel(excel_file)
aux_categ_cert_umc.head(2)

,ID_SSFF,No,PREGUNTA,N° Respuesta,Respuesta,Es correcto,Cargo,Temas,Familia,Fecha,Cargo_aux,Canal
0,UC_EYN_CERT_FORM_CC_AGO_2025,1,"Pregunta de única respuesta: Sebastián Ríos, c...",1.0,Confirmar que el cliente tenga un plan prepago...,False,CCE,Claro Sync,Móvil,2025-08-01,CCE,CC
1,UC_EYN_CERT_FORM_CC_AGO_2025,1,"Pregunta de única respuesta: Sebastián Ríos, c...",2.0,Validar que el reloj esté conectado por Blueto...,False,CCE,Claro Sync,Móvil,2025-08-01,CCE,CC


In [5]:
# Intentar convertir todas las columnas a pandas StringDtype; si falla, hacer fallback a str
try:
    aux_categ_cert_umc = aux_categ_cert_umc.astype({col: "string" for col in aux_categ_cert_umc.columns})
except Exception:
    aux_categ_cert_umc = aux_categ_cert_umc.applymap(lambda x: "" if pd.isna(x) else str(x))

In [6]:
aux_categ_cert_umc.to_parquet(
'data/listos_para_cargar_databricks/aux_categ_cert_umc.parquet',
engine='pyarrow',
index=False,
coerce_timestamps='ms',
allow_truncated_timestamps=True
)
print("DataFrame exported successfully to Parquet format")

DataFrame exported successfully to Parquet format
